In [ ]:
!pip install plotly
!pip install scipy

### Normalize CSV

In [ ]:
import pandas as pd

df = pd.read_csv('4245.csv') #attach CSV from graph data extractor or download from a sink

min_t = df['tick'].min()
max_t = df['tick'].max()

pivot_df = df.groupby(['wire_id', 'tick'])['adc_value'].max().unstack(fill_value=0)

all_ticks = list(range(min_t, max_t + 1))

pivot_df = pivot_df.reindex(columns=all_ticks, fill_value=0)
pivot_df = pivot_df.sort_index(ascending=False)

print(f"Time range: {min_t} to {max_t}")
print(df['wire_id'].max())
pivot_df.head(50)

Below are the notebook snippets to generate graphs; you could plug these into functions if you wanted to repeatedly regenerate the graph, but it would be quite slow.

### Generate Heatmap

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

data = []
wire_ids = pivot_df.index.tolist()
ticks = np.arange(min_t, max_t) #range of all ticks that are found in the csv

for wire_id in wire_ids:
    row_values = pivot_df.loc[wire_id].values[0:max_t]
    data.append(row_values)
    
data_array = np.array(data)
wire_pedestals = np.median(data_array, axis=1, keepdims=True)
data_subtracted = data_array - wire_pedestals #normalize

#set up heatmap figure
fig = go.Figure(data=go.Heatmap(
    z=data_subtracted,
    x=ticks,      
    y=wire_ids,
    zmax=50,
    zmin=-20,
    colorscale='Thermal',
    hovertemplate=(
        "<b>Wire: %{y}</b><br>" +
        "Tick: %{x}<br>" +
        "Chan_value: %{z:.2f}" +
        "<extra></extra>"
    ),
    
    colorbar=dict(title="ADC Value", thickness=15)
))

#format layout
fig.update_layout(
    title='Collection Plane Heatmap TPC=0',
    xaxis_title='Time (Tick)',
    yaxis_title='Wire #',
    plot_bgcolor='rgb(10, 10, 10)',
    paper_bgcolor='rgb(10, 10, 10)',
    font=dict(color='white'),
    xaxis=dict(range=[min_t, max_t], showgrid=False),
    yaxis=dict(
        type='category',   
        tickmode='linear',
        automargin=True,
        autorange='reversed'
    ),
    font_size = 8,
    hovermode='closest',
    height=1300,
    width=900
)

fig.show()

### Generate Ridgeline Graph

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# Adjust graph
signal_gain = .01 #Adjust for steepness of peaks
step = 1.0 

fig = go.Figure()

wire_ids = pivot_df.index.tolist()[::-1]
n_wires = len(wire_ids)
ticks = np.arange(min_t,max_t)
colorscale = px.colors.cyclical.Phase

for i, wire_id in enumerate(wire_ids):
    raw_values = pivot_df.loc[wire_id].values[0:max_t]
    baseline = i * step

    y_vals = (raw_values - np.mean(raw_values)) * signal_gain + baseline
    
    color_index = int(i / n_wires * (len(colorscale) - 1))
    line_color = colorscale[color_index]
    
    fig.add_trace(go.Scatter(
        x=ticks,
        y=y_vals,
        customdata=raw_values, 
        line=dict(color=line_color, width=1.2),
        name=f'Wire {wire_id}',
        mode='lines',
        hovertemplate=(
            "<b>Wire %{fullData.name}</b><br>" +
            "Time (Tick): %{x}<br>" +
            "ADC Value: %{customdata}<extra></extra>"
        ),
        showlegend=False
    ))

fig.update_layout(
    title='Collection Plane Activity (Hover to see Raw ADC)',
    xaxis_title='Time (Tick)',
    yaxis_title='Wire ID',
    hovermode='closest', 
    yaxis=dict(
        tickmode='array',
        tickvals=[i * step for i in range(n_wires)],
        ticktext=wire_ids,
        showgrid=True,
        gridcolor='rgba(255,255,255,0.05)',
        zeroline=False,
    ),
    xaxis=dict(range=[min_t, max_t], showgrid=False),
    plot_bgcolor='rgb(10, 10, 10)',
    paper_bgcolor='rgb(10, 10, 10)',
    font=dict(color='white'),
    height=2500,
    width=1200,
    margin=dict(l=50, r=20, t=50, b=50)
)

fig.show()